# Comparison of models to test explanation of CE signatures

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join('data', 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join('data', 'final-document.tsv')
METADATA_DOC = os.path.join('data', 'all-candor-meta_data.csv')

#### Import CEDA ready doc

In [3]:
df = pd.read_table(FINAL_DOC, sep='\t')
print(df.shape)
df.head()

(281817, 18)


,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,groups,dyad_,resid,x_conv_leader,x_in_common,x_responsive_1,x_responsive_2,x_responsive_3,x_you_are_good_listener
0,2.277085,13.0,15.0,1,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,3482,-0.456335,5.0,3.0,5.0,5.0,6.0,7.0
1,2.241708,13.0,16.0,3,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,3482,-0.479716,5.0,3.0,5.0,5.0,6.0,7.0
2,2.280164,13.0,26.0,5,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,3482,-0.317505,5.0,3.0,5.0,5.0,6.0,7.0
3,1.859337,13.0,17.0,7,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,3482,-0.850510,5.0,3.0,5.0,5.0,6.0,7.0
4,2.359147,13.0,6.0,9,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,3482,-0.487716,5.0,3.0,5.0,5.0,6.0,7.0


#### Import CANDOR Metadata

In [4]:
dfm = pd.read_csv(METADATA_DOC)
print(dfm.shape)
dfm.head()

(68, 222)


,user_id,partner_id,convo_id,date,survey_duration_in_seconds,time_zone,pre_affect,pre_arousal,technical_quality,conv_length,...,my_conscientious,my_neurotic,my_open,your_extraversion,your_agreeable,your_conscientious,your_neurotic,your_open,who_i_talked_to_most_past24,most_common_format_past24
0,5efbe4f0c9f37c1a3b70afaf,5f26378609aca228dcfd19fa,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,2020-11-10,5515,5.0,7.0,6.0,1.0,29.0,...,3.333333,4.333333,4.666667,3.333333,4.333333,3.666667,3.000000,4.000000,NaN,NaN
1,5f26378609aca228dcfd19fa,5efbe4f0c9f37c1a3b70afaf,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,2020-11-10,2507,8.0,6.0,6.0,1.0,30.0,...,4.000000,2.666667,3.666667,2.666667,4.666667,5.000000,2.666667,4.333333,NaN,NaN
2,5db721e8e26da7000b6c7f85,5eeb412dde2ce12142b917b3,f84ed43d-38b0-4f89-aca3-32bd12d8e4fa,2020-06-30,2852,5.0,4.0,3.0,2.0,25.0,...,3.333333,3.666667,4.666667,3.333333,3.000000,3.333333,3.666667,3.333333,friends,video_chat
3,5eeb412dde2ce12142b917b3,5db721e8e26da7000b6c7f85,f84ed43d-38b0-4f89-aca3-32bd12d8e4fa,2020-06-30,3943,8.0,5.0,5.0,2.0,25.0,...,3.000000,2.333333,4.333333,3.333333,3.666667,3.666667,2.333333,4.333333,NaN,NaN
4,5e97478f8b670f09dad1fd02,5f13bae4a544193e6b94b0cf,4e02e612-69a5-4323-8610-e89822241fc2,2020-07-31,4757,5.0,6.0,3.0,1.0,25.0,...,2.666667,3.333333,3.666667,3.666667,3.333333,3.333333,3.000000,3.000000,NaN,NaN


## Selecting Columns and adding them

In [5]:
addl_cols = [
    'conv_leader',
    'in_common',
    'responsive_1',
    'responsive_2',
    'responsive_3',
    'you_are_good_listener',
    'our_thoughts_synced_up_sr1',
    'developed_joint_perspective_sr2',
    'thoughts_became_more_alike_sr5',
    'saw_world_in_same_way_sr8',
    'you_are_good_listener'
]

In [6]:
conversion = {
    col: {
        uid: val 
        for uid,val in dfm[['user_id', col]].values
    } for col in addl_cols
}

In [7]:
for col, d in conversion.items():
    print(col)
    df['x_'+col] = [d[uid] for uid in tqdm(df['x_speaker'].values)]

conv_leader


  0%|          | 0/281817 [00:00<?, ?it/s]

in_common


  0%|          | 0/281817 [00:00<?, ?it/s]

responsive_1


...

  0%|          | 0/281817 [00:00<?, ?it/s]

thoughts_became_more_alike_sr5


  0%|          | 0/281817 [00:00<?, ?it/s]

saw_world_in_same_way_sr8


  0%|          | 0/281817 [00:00<?, ?it/s]

In [8]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,groups,...,x_conv_leader,x_in_common,x_responsive_1,x_responsive_2,x_responsive_3,x_you_are_good_listener,x_our_thoughts_synced_up_sr1,x_developed_joint_perspective_sr2,x_thoughts_became_more_alike_sr5,x_saw_world_in_same_way_sr8
0,2.277085,13.0,15.0,1,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,...,5.0,3.0,5.0,5.0,6.0,7.0,5.0,5.0,4.0,6.0
1,2.241708,13.0,16.0,3,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,...,5.0,3.0,5.0,5.0,6.0,7.0,5.0,5.0,4.0,6.0
2,2.280164,13.0,26.0,5,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,...,5.0,3.0,5.0,5.0,6.0,7.0,5.0,5.0,4.0,6.0
3,1.859337,13.0,17.0,7,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,...,5.0,3.0,5.0,5.0,6.0,7.0,5.0,5.0,4.0,6.0
4,2.359147,13.0,6.0,9,False,5e97478f8b670f09dad1fd02,5e97478f8b670f09dad1fd02-5f13bae4a544193e6b94b0cf,CANDOR,candor-2,data-5e97478f8b670f09dad1fd02,...,5.0,3.0,5.0,5.0,6.0,7.0,5.0,5.0,4.0,6.0


## Save outputs

In [9]:
df.to_csv(FINAL_DOC, index=False, encoding='utf-8', sep='\t')